<a href="https://colab.research.google.com/github/sanrodmu29/Curso_Programacion_Cientifica/blob/main/ejercicios/taller-evaluativo-03-numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller evaluativo 03 — NumPy
**Nombre completo:** [tu nombre aquí]
**Fecha:** [fecha aquí]

## Ejercicio 1 — Crear los arreglos y leer su etiqueta
Convierto la tabla de lecturas crudas en un arreglo 2D de NumPy, creo el arreglo de días con np.arange y los de calibración, y leo los atributos shape, ndim, size y dtype.

In [ ]:
import numpy as np

lecturas_crudas = np.array([
    [10.0, 14.5, 22.0, 30.5, 28.0, 18.0,  9.5],
    [12.0, 20.0, 35.0, 48.0, 41.0, 25.0, 14.0],
    [ 8.0, 11.0, 19.0, 27.0, 33.0, 21.0,  6.0],
    [15.0, 24.0, 31.0, 44.0, 52.0, 36.0, 17.0],
    [ 5.0,  9.0, 16.0, 22.0, 45.0, 30.0,  4.0],
])

forma = lecturas_crudas.shape
dimensiones = lecturas_crudas.ndim
total = lecturas_crudas.size
tipo = lecturas_crudas.dtype

dias = np.arange(1, 8)
pendientes = np.array([1.2, 0.9, 1.1, 1.0, 1.3])
interceptos = np.array([-0.5, 1.0, -2.0, 0.5, -1.5])

print(forma, dimensiones, total, tipo)
print(dias)
print(pendientes)
print(interceptos)

(5, 7) 2 35 float64
[1 2 3 4 5 6 7]
[1.2 0.9 1.1 1.  1.3]
[-0.5  1.  -2.   0.5 -1.5]


## Ejercicio 2 — Señalar celdas y recortar bloques
Uso indexación y slicing sobre lecturas_crudas para leer una celda, usar un índice negativo, extraer una columna completa, una fila completa y un bloque de filas y columnas.

In [ ]:
lectura_c_dia4 = lecturas_crudas[2, 3]
ultimo_primer = lecturas_crudas[-1, 0]
dia5 = lecturas_crudas[:, 4]
semana_b = lecturas_crudas[1, :]
bloque = lecturas_crudas[1:4, 2:6]

print(lectura_c_dia4)
print(ultimo_primer)
print(dia5)
print(semana_b)
print(bloque)
print(bloque.shape)

27.0
5.0
[28. 41. 33. 52. 45.]
[12. 20. 35. 48. 41. 25. 14.]
[[35. 48. 41. 25.]
 [19. 27. 33. 21.]
 [31. 44. 52. 36.]]
(3, 4)


## Ejercicio 3 — Limpiar una campaña con máscaras booleanas
Construyo la campaña de SBC-A, creo una máscara del valor centinela -999.0 y otra de lecturas útiles (0 a 500.0), filtro con la máscara y calculo el promedio.

In [ ]:
campana_sbc_a = np.array([10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0])

faltantes = campana_sbc_a == -999.0
es_util = (campana_sbc_a >= 0) & (campana_sbc_a <= 500.0)
lecturas_utiles = campana_sbc_a[es_util]
promedio_utiles = lecturas_utiles.mean()

print(int(faltantes.sum()))
print(lecturas_utiles)
print(promedio_utiles)

2
[10. 25. 40.  8. 52.]
27.0


## Ejercicio 4 — Calibrar todo con operaciones vectorizadas y broadcasting

¿Por qué lecturas_crudas * pendientes lanza ValueError? Porque NumPy alinea las formas de derecha a izquierda: lecturas_crudas tiene forma (5, 7) y pendientes forma (5,). El 5 de pendientes se compara con el 7 de las columnas y no coinciden, así que el broadcasting falla. La solución es convertir pendientes e interceptos en columnas de forma (5, 1): así el eje de 1 se estira sobre los 7 días y la operación se realiza sin bucles.

In [ ]:
pendientes_col = pendientes.reshape(5, 1)
interceptos_col = interceptos.reshape(5, 1)

calibradas = lecturas_crudas * pendientes_col + interceptos_col

campana_calibrada = lecturas_utiles * 1.2 - 0.5
promedio_calibrado_sbc_a = campana_calibrada.mean()

print(calibradas.shape)
print(calibradas[0])
print(campana_calibrada)
print(promedio_calibrado_sbc_a)

(5, 7)
[11.5 16.9 25.9 36.1 33.1 21.1 10.9]
[11.5 29.5 47.5  9.1 61.9]
31.9


## Ejercicio 5 — Resumir la semana con estadística por eje
Con calibradas calculo el promedio general, el promedio por sensor (axis=1), el promedio por día (axis=0), el máximo por sensor y la media de las lecturas que superan 35.4, redondeando con np.round.

In [ ]:
promedio_general = calibradas.mean()
promedio_por_sensor = calibradas.mean(axis=1)
promedio_por_dia = calibradas.mean(axis=0)
maximo_por_sensor = calibradas.max(axis=1)
media_sobre_limite = calibradas[calibradas > 35.4].mean()

print(np.round(promedio_general, 2))
print(np.round(promedio_por_sensor, 2))
print(np.round(promedio_por_dia, 2))
print(np.round(maximo_por_sensor, 2))
print(np.round(media_sobre_limite, 2))

24.11
[22.21 26.07 17.64 31.79 22.83]
[10.12 16.14 25.62 35.92 42.96 27.94 10.06]
[36.1 44.2 34.3 52.5 57. ]
43.28


## Ejercicio 6 — Funciones universales y reemplazo con máscara
Uso np.maximum para medir cuánto excede cada lectura el límite de 35.4, sumo el excedente por sensor, calculo el desvío absoluto de SBC-B con np.abs y creo una copia en_alerta reemplazando con una máscara las lecturas que no superan 35.4.

In [ ]:
excedente = np.maximum(calibradas - 35.4, 0)
excedente_por_sensor = np.round(excedente.sum(axis=1), 1)
desvio_sensor_b = np.round(np.abs(calibradas[1] - 35.4), 1)

en_alerta = calibradas.copy()
en_alerta[en_alerta <= 35.4] = 0.0

print(excedente_por_sensor)
print(desvio_sensor_b)
print(int((en_alerta > 0).sum()))

[ 0.7 11.3  0.  27.3 23.7]
[23.6 16.4  2.9  8.8  2.5 11.9 21.8]
8


## Ejercicio 7 — Integración: el informe de la red
Con máscaras sobre calibradas cuento cuántas lecturas caen en cada categoría de la escala oficial y encuentro el día más contaminado filtrando dias con una máscara sobre promedio_por_dia.

In [ ]:
buenas = (calibradas <= 12.0).sum()
moderadas = ((calibradas > 12.0) & (calibradas <= 35.4)).sum()
daninas_sensibles = ((calibradas > 35.4) & (calibradas <= 55.4)).sum()
daninas_salud = (calibradas > 55.4).sum()

dia_pico = dias[promedio_por_dia == promedio_por_dia.max()][0]

print(f"Lecturas por categoría: Buena={buenas}, Moderada={moderadas}, Dañina para grupos sensibles={daninas_sensibles}, Dañina para la salud={daninas_salud}")
print(f"Día más contaminado: {dia_pico} con {np.round(promedio_por_dia.max(), 1)} ug/m3")

Lecturas por categoría: Buena=9, Moderada=18, Dañina para grupos sensibles=7, Dañina para la salud=1
Día más contaminado: 5 con 43.0 ug/m3


In [ ]:
# Celda de verificación — ejecútela al final, después de todo lo demás
assert lecturas_crudas.shape == (5, 7) and lecturas_crudas.ndim == 2, "revise lecturas_crudas del Ejercicio 1"
assert forma == (5, 7) and dimensiones == 2 and total == 35 and tipo == np.float64, "revise los atributos del Ejercicio 1"
assert np.array_equal(dias, np.arange(1, 8)), "dias debe ir del 1 al 7 (el 7 incluido)"
assert np.allclose(pendientes, [1.2, 0.9, 1.1, 1.0, 1.3]) and np.allclose(interceptos, [-0.5, 1.0, -2.0, 0.5, -1.5]), "revise pendientes e interceptos"
assert lectura_c_dia4 == 27.0 and ultimo_primer == 5.0, "revise la indexación del Ejercicio 2"
assert np.array_equal(dia5, [28.0, 41.0, 33.0, 52.0, 45.0]) and semana_b.shape == (7,), "revise dia5 y semana_b del Ejercicio 2"
assert bloque.shape == (3, 4) and bloque[0, 0] == 35.0, "revise el bloque del Ejercicio 2"
assert int(faltantes.sum()) == 2 and lecturas_utiles.shape == (5,), "revise las máscaras del Ejercicio 3"
assert abs(promedio_utiles - 27.0) < 1e-9, "revise promedio_utiles del Ejercicio 3"
assert calibradas.shape == (5, 7), "calibradas debe tener forma (5, 7)"
assert np.allclose(calibradas[0], [11.5, 16.9, 25.9, 36.1, 33.1, 21.1, 10.9]), "revise la calibración del Ejercicio 4"
assert lecturas_crudas[0, 0] == 10.0, "lecturas_crudas no debe modificarse"
assert abs(promedio_calibrado_sbc_a - 31.9) < 1e-9, "revise promedio_calibrado_sbc_a del Ejercicio 4"
assert abs(promedio_general - 24.1086) < 1e-4, "revise promedio_general del Ejercicio 5"
assert np.allclose(np.round(promedio_por_sensor, 2), [22.21, 26.07, 17.64, 31.79, 22.83]), "revise promedio_por_sensor (axis=1)"
assert np.allclose(np.round(promedio_por_dia, 2), [10.12, 16.14, 25.62, 35.92, 42.96, 27.94, 10.06]), "revise promedio_por_dia (axis=0)"
assert np.allclose(maximo_por_sensor, [36.1, 44.2, 34.3, 52.5, 57.0]), "revise maximo_por_sensor"
assert abs(media_sobre_limite - 43.275) < 1e-9, "revise media_sobre_limite"
assert np.allclose(excedente_por_sensor, [0.7, 11.3, 0.0, 27.3, 23.7]), "revise excedente_por_sensor del Ejercicio 6"
assert np.allclose(desvio_sensor_b, [23.6, 16.4, 2.9, 8.8, 2.5, 11.9, 21.8]), "revise desvio_sensor_b"
assert en_alerta.shape == (5, 7) and int((en_alerta > 0).sum()) == 8, "revise en_alerta"
assert abs(calibradas[0, 3] - 36.1) < 1e-9, "calibradas no debe modificarse al crear en_alerta"
assert (buenas, moderadas, daninas_sensibles, daninas_salud) == (9, 18, 7, 1), "revise los conteos del Ejercicio 7"
assert buenas + moderadas + daninas_sensibles + daninas_salud == calibradas.size, "los cuatro conteos deben sumar 35"
assert dia_pico == 5, "revise dia_pico del Ejercicio 7"
print("Verificación completada sin errores.")

Verificación completada sin errores.
